In [11]:
# pip install mediapipe 

In [ ]:
import cv2
import mediapipe as mp

# Import new API
from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# Load model
base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert to MediaPipe Image
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    result = detector.detect(mp_image)

    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            for lm in hand:
                h, w, _ = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0,255,0), -1)

    cv2.imshow("Hand Detection (New API)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp

from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# Load model
base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

# ✅ Define hand connections (same as old MediaPipe)
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),        # Thumb
    (0,5),(5,6),(6,7),(7,8),        # Index
    (0,9),(9,10),(10,11),(11,12),   # Middle
    (0,13),(13,14),(14,15),(15,16), # Ring
    (0,17),(17,18),(18,19),(19,20)  # Pinky
]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = detector.detect(mp_image)

    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            h, w, _ = frame.shape

            # ✅ Store all landmark points
            points = []
            for lm in hand:
                cx, cy = int(lm.x * w), int(lm.y * h)
                points.append((cx, cy))

                # Draw points
                cv2.circle(frame, (cx, cy), 5, (0,255,0), -1)

            # ✅ Draw connections (LINES)
            for connection in HAND_CONNECTIONS:
                start_idx, end_idx = connection
                cv2.line(frame, points[start_idx], points[end_idx], (255,0,0), 2)

    cv2.imshow("Hand Detection (with Lines)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp

from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# Load model
base_options = python.BaseOptions(model_asset_path="pose_landmarker_full.task")

options = vision.PoseLandmarkerOptions(
    base_options=base_options
)

detector = vision.PoseLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

# ✅ Pose connections (skeleton structure)
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),      # Right arm
    (0,4),(4,5),(5,6),(6,8),      # Left arm
    (9,10),                       # Shoulders
    (11,12),                      # Hips
    (11,13),(13,15),(15,17),(17,19),(19,15),  # Left leg
    (12,14),(14,16),(16,18),(18,20),(20,16),  # Right leg
    (11,23),(12,24),(23,24)       # Torso
]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if result.pose_landmarks:
        h, w, _ = frame.shape

        # ✅ Store all landmark points
        points = []
        for lm in result.pose_landmarks[0]:
            cx, cy = int(lm.x * w), int(lm.y * h)
            points.append((cx, cy))

            # Draw points
            cv2.circle(frame, (cx, cy), 5, (0,255,0), -1)

        # ✅ Draw connections (LINES)
        for connection in POSE_CONNECTIONS:
            start_idx, end_idx = connection

            # Safety check (avoid index errors)
            if start_idx < len(points) and end_idx < len(points):
                cv2.line(frame, points[start_idx], points[end_idx], (255,0,0), 2)

    cv2.imshow("Pose (with Lines)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp

from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# Load model
base_options = python.BaseOptions(model_asset_path="face_landmarker.task")

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

detector = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

# ✅ Face mesh connections (simplified for clarity)
# Full mesh is large → here are key regions
FACE_CONNECTIONS = [
    # Jaw
    (152, 148),(148, 176),(176, 149),(149, 150),(150, 136),
    # Left eye
    (33, 7),(7, 163),(163, 144),(144, 145),(145, 153),(153, 154),(154, 155),(155, 133),
    # Right eye
    (362, 382),(382, 381),(381, 380),(380, 374),(374, 373),(373, 390),(390, 249),(249, 263),
    # Lips
    (61,146),(146,91),(91,181),(181,84),(84,17),(17,314),(314,405),(405,321),(321,375),(375,291)
]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if result.face_landmarks:
        h, w, _ = frame.shape

        for face in result.face_landmarks:
            points = []

            # ✅ Draw points
            for lm in face:
                cx, cy = int(lm.x * w), int(lm.y * h)
                points.append((cx, cy))
                cv2.circle(frame, (cx, cy), 1, (0,255,0), -1)

            # ✅ Draw connections
            for (start, end) in FACE_CONNECTIONS:
                if start < len(points) and end < len(points):
                    cv2.line(frame, points[start], points[end], (255,0,0), 1)

    cv2.imshow("Face Mesh (New API)", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()